In [1]:
# CELL 0 – Đọc config và tạo biến FEATURES, RESULTS
import os, json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, ttest_ind

# notebooks/ nằm trong thư mục PROJECT_ROOT/notebooks
PROJECT_ROOT = Path(os.getcwd()).parent
CONFIG_PATH = PROJECT_ROOT / "src" / "config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy {CONFIG_PATH}. Hãy chạy 00_prep_features.ipynb trước.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = json.load(f)

FEATURES = Path(cfg["FEATURES"])
RESULTS  = Path(cfg["RESULTS"])

# Đảm bảo thư mục results tồn tại
RESULTS.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURES:", FEATURES)
print("RESULTS :", RESULTS)


PROJECT_ROOT: D:\STAT3013.Q12_Group01
FEATURES: D:\STAT3013.Q12_Group01\features
RESULTS : D:\STAT3013.Q12_Group01\results


In [2]:
# CELL 1 – Đọc dữ liệu repeat30d.parquet
import pandas as pd

orders_path = FEATURES / "repeat30d.parquet"
print("Đang kiểm tra file:", orders_path)

if orders_path.exists():
    orders = pd.read_parquet(orders_path)
    print(f"✅ Đã tải dữ liệu 'orders': {orders.shape}")
else:
    raise FileNotFoundError(
        f"❌ Không tìm thấy file tại: {orders_path}. Hãy chạy NB02 (build_features_repeat30d) trước!"
    )

orders.head()


Đang kiểm tra file: D:\STAT3013.Q12_Group01\features\repeat30d.parquet
✅ Đã tải dữ liệu 'orders': (113574, 28)


,household_key,basket_id,day,week_no,store_id,basket_value,basket_qty,retail_disc,coupon_disc,coupon_match_disc,...,gap_days,repeat30d,prev_day,recency,frequency,monetary_mean_prev,tenure,dow,weekofyear,discount_rate
0,1,27601281299,51,8,436,78.66,34,-16.54,-1.0,0.0,...,16.0,1,NaN,999.0,0,78.660000,0,2,8,0.182328
1,1,27774192959,67,10,436,41.10,14,-8.59,0.0,0.0,...,21.0,1,51.0,16.0,1,78.660000,16,4,10,0.172872
2,1,28024266849,88,13,436,26.90,13,-6.72,0.0,0.0,...,6.0,1,67.0,21.0,2,59.880000,37,4,13,0.199881
3,1,28106322445,94,14,436,63.43,32,-11.08,-0.5,-0.5,...,7.0,1,88.0,6.0,3,48.886667,43,3,14,0.159979
4,1,28235481967,101,15,436,53.45,20,-16.42,0.0,0.0,...,7.0,1,94.0,7.0,4,52.522500,50,3,15,0.235008


In [3]:
# CHI-SQUARE TEST (Treatment vs Repeat)
ct = pd.crosstab(orders["treatment"], orders["repeat30d"])
chi2, p_chi, dof, exp = chi2_contingency(ct)

# Cramer's V (Effect size cho Chi-square)
cramers_v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))

print("=== Chi-square Result ===")
print(f"Chi2: {chi2:.4f}, p-value: {p_chi:.6f}, Cramer's V: {cramers_v:.4f}")


# T-TEST (So sánh Basket Value - Cân bằng theo tuần)
# Lý do cân bằng: Tránh thiên lệch do số lượng đơn hàng các tuần khác nhau
treated = orders[orders.treatment == 1]
control = orders[orders.treatment == 0]

vals_t, vals_c = [], []
for w in orders.weekofyear.unique():
    vt = treated[treated.weekofyear == w]["basket_value"]
    vc = control[control.weekofyear == w]["basket_value"]
    
    # Chỉ lấy mẫu nếu cả 2 nhóm đều có đủ dữ liệu (>10 mẫu)
    if len(vt) > 10 and len(vc) > 10:
        n = min(len(vt), len(vc))
        vals_t.append(vt.sample(n, random_state=42))
        vals_c.append(vc.sample(n, random_state=42))

if len(vals_t) > 0:
    vals_t = pd.concat(vals_t)
    vals_c = pd.concat(vals_c)
    
    x = vals_t
    y = vals_c
    
    mean_t, mean_c = x.mean(), y.mean()
    std_t, std_c   = x.std(ddof=1), y.std(ddof=1)
    n_t, n_c       = len(x), len(y)

    tstat, p_t = ttest_ind(x, y, equal_var=False)

    s_pooled = np.sqrt(((n_t - 1) * std_t**2 + (n_c - 1) * std_c**2) / (n_t + n_c - 2))

    # Cohen's d
    cohens_d = (mean_t - mean_c) / s_pooled

    # Hedges' g (Bias-corrected)
    J = 1 - (3 / (4 * (n_t + n_c) - 9))
    hedges_g = cohens_d * J

    print("\n=== T-test Result ===")
    print(f"Mean Treated: {mean_t:.2f} | Mean Control: {mean_c:.2f}")
    print(f"T-stat: {tstat:.4f} | p-value: {p_t:.6f}")
    print(f"Cohen's d: {cohens_d:.4f} | Hedges' g: {hedges_g:.4f}")

else:
    print("Cảnh báo: Không đủ dữ liệu để chạy T-test cân bằng.")
    # Gán giá trị mặc định để tránh lỗi ở cell lưu kết quả phía sau
    tstat, p_t, cohens_d, hedges_g = 0, 1, 0, 0
    mean_t, mean_c, std_t, std_c, n_t, n_c = 0, 0, 0, 0, 0, 0

=== Chi-square Result ===
Chi2: 0.0109, p-value: 0.916665, Cramer's V: 0.0003

=== T-test Result ===
Mean Treated: 68.65 | Mean Control: 26.24
T-stat: 46.7818 | p-value: 0.000000
Cohen's d: 0.9082 | Hedges' g: 0.9081


In [4]:
import json
RESULTS.mkdir(exist_ok=True)

metrics_json = {
    "chi_square": {
        "p_value": float(p_chi),
        "cramers_v": float(cramers_v)
    },
    "t_test": {
        "p_value": float(p_t),
        "cohens_d": float(cohens_d),
        "hedges_g": float(hedges_g),
        "mean_lift": float(mean_t - mean_c)
    }
}

metrics_path = RESULTS / "nb04_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_json, f, ensure_ascii=False, indent=2)

print("Đã lưu metrics JSON vào:", metrics_path)

contingency_path = RESULTS / "nb04_contingency_table.csv"
ct.to_csv(contingency_path)
print("Đã lưu bảng contingency vào:", contingency_path)

chi_df = pd.DataFrame([{
    "chi2": chi2,
    "dof": dof,
    "p_value": p_chi,
    "cramers_v": cramers_v,
    "n_obs": int(ct.values.sum())
}])
chi_path = RESULTS / "nb04_chi_square.csv"
chi_df.to_csv(chi_path, index=False)
print("Đã lưu kết quả Chi-square vào:", chi_path)

tt_df = pd.DataFrame([{
    "mean_treated": mean_t,
    "mean_control": mean_c,
    "std_treated": std_t,
    "std_control": std_c,
    "n_treated": n_t,
    "n_control": n_c,
    "t_stat": tstat,
    "p_value": p_t,
    "cohens_d": cohens_d,
    "hedges_g": hedges_g
}])
tt_path = RESULTS / "nb04_ttest_effect_size.csv"
tt_df.to_csv(tt_path, index=False)
print("Đã lưu kết quả T-test vào:", tt_path)

tt_df


Đã lưu metrics JSON vào: D:\STAT3013.Q12_Group01\results\nb04_metrics.json
Đã lưu bảng contingency vào: D:\STAT3013.Q12_Group01\results\nb04_contingency_table.csv
Đã lưu kết quả Chi-square vào: D:\STAT3013.Q12_Group01\results\nb04_chi_square.csv
Đã lưu kết quả T-test vào: D:\STAT3013.Q12_Group01\results\nb04_ttest_effect_size.csv


,mean_treated,mean_control,std_treated,std_control,n_treated,n_control,t_stat,p_value,cohens_d,hedges_g
0,68.646998,26.238994,57.320356,32.79371,5307,5307,46.781788,0.0,0.90817,0.908106
